<!--
SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
Copyright (c) 2026 Intel Corporation
SPDX-License-Identifier: Apache-2.0

NOTE: This file has been modified by Intel Corporation.
-->

# Notebook goal
The purpose of this notebook is to demo Torch Tweak module inspecting and automated wrapping.

In [ ]:
%load_ext autoreload
%autoreload 2
%cd ..
%pwd

In [ ]:
import torch_tweak.torch as tt

# Inspecting Stable Diffusion pipeline

Let's use Stable Diffusion 3 as example pipeline for inspect and automated wrapping.

In [ ]:
import diffusers

model_name = "stable-diffusion-v1-5/stable-diffusion-v1-5"
pipe = diffusers.DiffusionPipeline.from_pretrained(model_name)

pipe.to("xpu")

Inspect the pipeline on sample input prompt:

In [ ]:
input_data = [{"prompt": "A beautiful landscape with mountains and a lake"}]

modules_info = tt.inspect(pipe, input_data)

Once the pipeline has been evaluated we got collected information about the top level executed modules within the pipeline. First, let's see the detail information about the modules:

In [ ]:
modules_info.describe()

Example output:

In [ ]:
Module Execution Summary:
==========================================================================================================================================
    Module Name        Calls    Total Time (s)    Avg Time (s)    % of Total    # of params      # of layers           precisions        
------------------------------------------------------------------------------------------------------------------------------------------
decoder                     10           2.7948           0.2795        2.25%         49490179                6              torch.float32
post_quant_conv             10           0.0029           0.0003        0.00%               20                0              torch.float32
text_encoder                20           0.1761           0.0088        0.14%        123060480                3              torch.float32
unet                       510         119.9169           0.2351       96.59%        859520964                9              torch.float32
safety_checker              10           0.2158           0.0216        0.17%        303981588                2              torch.float32
------------------------------------------------------------------------------------------------------------------------------------------
Total execution time: 124.146304 seconds
Number of batch iterations: 10
==========================================================================================================================================

The table present information about each module with details:
- how many times each module has been executed
- what was the total inference time that was spent during inference call in each module
- what was the average time of a single module forward call
- what fraction of the total pipeline runtime was spent inside this module

And some more information about number of parameters, layers and precision of each module.

# Using inspect information for tuning

Once the information about modules was collected we can move to wrapping the modules for further tuning. Here we have couple of options that could be applied for selecting the modules for tuning.

### Selecting all modules

First and simplest is selecting all modules for tuning with following code:

In [ ]:
modules = modules_info.get_modules()

### Selecting modules conditionally

In certain situation we would like to wrap only the top executed modules based on some conditions like:
- top k modules based on % of total execution time
- minimal % of total execution time

For such purpose use a different method:

In [ ]:
# Top 1 module
modules = modules_info.get_modules(limit=1)

# Minimum 40% of total execution time
modules = modules_info.get_modules(min_execution_percentage=0.4)

### Wrapping and tunning modules

Once we have selected modules we can automatically wrap them and perform tuning:

In [ ]:
from torch_tweak.torch.backend import OpenVINOBackend, TorchInductorBackend
from torch_tweak.torch.tune_strategy import FirstWinsStrategy

# Wrap modules
pipe = tt.wrap(
    pipe, modules, strategy=FirstWinsStrategy(backends=[OpenVINOBackend(), TorchInductorBackend()])
)

Finally, execute tune:

In [ ]:
# Tune modules
pipe = tt.tune(pipe, input_data, batch_sizes=[1])